# GraphRAG

A refresher on **GraphRAG** — retrieval-augmented generation that retrieves over a
**knowledge graph of entities and relationships** built from your corpus, rather than over a
flat pile of text chunks. Popularised by Microsoft Research's *GraphRAG* (2024), its signature
win is **global, "sensemaking" questions** ("what are the main themes across these documents?")
that vanilla vector RAG cannot answer.

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

Vanilla RAG embeds your documents into chunks, retrieves the top-`k` most similar chunks to a
question, and stuffs them into the prompt. It is excellent at **local** questions whose answer
sits in one or two chunks ("what is the refund window?"). It is bad at two things:

1. **Multi-hop questions** — answers that require *connecting* facts spread across documents
   ("which of our suppliers is owned by a company we're suing?"). The bridging chunk may not be
   similar to the question at all, so it never gets retrieved.
2. **Global / sensemaking questions** — "what are the recurring themes in these 500 incident
   reports?" No single chunk contains the answer; it is a property of the *whole* corpus.

**GraphRAG** attacks both by adding a structured layer. An indexing pass uses an LLM to extract
**entities** (nodes) and **relationships** (edges) from every chunk, building a knowledge graph.
It then runs **community detection** to cluster the graph into themes and pre-generates a
**summary for each community**. At query time you get two retrieval modes:

- **Local search** — start from entities mentioned in the question, walk their neighbourhood, and
  feed the connected sub-graph + source text to the LLM. This is what answers multi-hop queries.
- **Global search** — map the question over the community summaries, then reduce the partial
  answers into one. This is what answers sensemaking queries.

Reach for GraphRAG when relationships *between* facts matter, or when users ask corpus-wide
questions. Skip it when your questions are local lookups — the indexing cost (an LLM call per
chunk) is real and a good vector store will be cheaper and just as accurate.

## 2. Mental Model

**Vanilla RAG is a librarian who hands you the few pages that look most like your question.
GraphRAG first reads every book and draws a *concept map* on the wall — who relates to whom, which
topics cluster together — then answers by tracing lines on that map.**

- **Local search ≈ "follow the edges."** Find the entity you asked about, then walk outward one or
  two hops to pull in connected entities the question never named. Vector RAG can't make that jump
  because the bridging fact isn't textually similar to the query.
- **Global search ≈ "read the chapter summaries."** The graph is partitioned into communities
  (clusters of densely connected entities ≈ themes); each has a pre-written summary. A corpus-wide
  question is answered by combining those summaries — *map* each summary to a partial answer, then
  *reduce* the partials into one. You never re-read the raw documents.

The graph is the index. Building it is expensive and offline; querying it is cheap and online.

## 3. Key Concepts

- **Entity / relationship extraction** — the indexing LLM reads each chunk and emits
  `(source, relation, target)` triples plus short entity descriptions. This is the costly step:
  one (or several) LLM calls **per chunk**.
- **Knowledge graph** — nodes are entities, edges are relationships. Edges and nodes carry
  descriptions and **provenance** (which chunk they came from) so answers can cite sources.
- **Community detection** — a graph-clustering algorithm (Microsoft uses **Leiden**; Louvain and
  greedy modularity are cousins) partitions nodes into communities that are densely linked inside
  and sparsely linked outside. Communities ≈ themes, and they are **hierarchical** (communities of
  communities).
- **Community report / summary** — an LLM-written summary of each community, generated at index
  time. These are what global search reads instead of raw text.
- **Local search** — entity-centric retrieval: seed from question entities, expand the
  neighbourhood, rank, and pass the sub-graph + linked text units to the LLM.
- **Global search (map-reduce)** — answer over community summaries: *map* the query against each
  summary to get scored partial answers, then *reduce* them into a final answer.
- **Provenance / grounding** — every node, edge, and report tracks its source chunks, so generated
  answers stay citable — the same grounding discipline as vanilla RAG.

## 4. Setup

The worked examples build and query a tiny knowledge graph by hand so the mechanism is fully
visible. They need only **NetworkX** (pure-Python graph library, CPU, no downloads):

```bash
%pip install networkx
```

The optional §5.3 cell shows real LLM-based triple extraction. It is **gated behind
`ANTHROPIC_API_KEY`** — without a key the notebook still runs end-to-end and prints the call shape
plus a regex fallback. To run it for real:

```bash
%pip install anthropic
export ANTHROPIC_API_KEY=sk-ant-...
```

The production system — Microsoft's library — is `pip install graphrag` and is driven from the CLI
(`graphrag index` / `graphrag query`); it expects an LLM endpoint and is **not** CPU-tiny, so we
reimplement the ideas here instead.

In [1]:
# Environment probe — what's available in THIS kernel (no downloads, no API key needed).
import importlib.util
import os
import sys

import networkx as nx

def have(mod: str) -> str:
    return "installed" if importlib.util.find_spec(mod) else "not installed"

print(f"python              : {sys.version.split()[0]}")
print(f"networkx            : {nx.__version__}")
print(f"anthropic           : {have('anthropic')}")
print(f"ANTHROPIC_API_KEY   : {'set' if os.getenv('ANTHROPIC_API_KEY') else 'not set'}")

python              : 3.13.7
networkx            : 3.6.1
anthropic           : not installed
ANTHROPIC_API_KEY   : not set


## 5. Worked Examples

### 5.1 Local search: connecting facts vanilla RAG would miss

Below is a small corpus already reduced to extracted triples (in §5.3 we show how an LLM produces
these). The question — *"Is any supplier connected to a company Acme is suing?"* — is a **two-hop**
query: the answer requires chaining `Acme --supplier--> Globex` with `Globex --owned_by-->
Initech` and `Acme --suing--> Initech`. No single chunk states the conclusion, so a flat top-`k`
retriever keyed on the question would likely never surface the bridge. Graph traversal finds it.

In [2]:
import networkx as nx

# (subject, relation, object, source_chunk) — what an extraction pass would emit.
triples = [
    ("Acme",   "supplier_of",   "widgets",  "doc1#0"),
    ("Globex", "supplier_to",   "Acme",     "doc1#1"),
    ("Globex", "owned_by",      "Initech",  "doc2#0"),
    ("Acme",   "suing",         "Initech",  "doc3#0"),
    ("Initech","headquartered",  "Boston",  "doc2#1"),
    ("Acme",   "headquartered",  "Denver",  "doc1#2"),
]

G = nx.DiGraph()
for s, r, o, src in triples:
    G.add_edge(s, o, relation=r, source=src)

print(f"graph: {G.number_of_nodes()} entities, {G.number_of_edges()} relationships")

# Local search: seed from the entities in the question, expand the neighbourhood.
seeds = ["Acme"]
# Treat relationships as connections regardless of direction for traversal.
UG = G.to_undirected()
neighbourhood = set(seeds)
for s in seeds:
    neighbourhood |= set(nx.single_source_shortest_path_length(UG, s, cutoff=2))

print("\n2-hop neighbourhood of Acme:", sorted(neighbourhood))

# The bridge: a supplier of Acme that is owned by a company Acme is suing.
sued = {o for _, o, d in G.edges(data=True) if d["relation"] == "suing"}
suppliers = {s for s, o, d in G.edges(data=True)
             if d["relation"] == "supplier_to" and o == "Acme"}
for sup in suppliers:
    owners = {o for _, o, d in G.out_edges(sup, data=True) if d["relation"] == "owned_by"}
    flagged = owners & sued
    if flagged:
        print(f"\nFLAG: supplier {sup!r} is owned by {flagged} — which Acme is suing.")
        path = nx.shortest_path(UG, sup, list(flagged)[0])
        print("      bridge path:", " -> ".join(path))

graph: 6 entities, 6 relationships

2-hop neighbourhood of Acme: ['Acme', 'Boston', 'Denver', 'Globex', 'Initech', 'widgets']

FLAG: supplier 'Globex' is owned by {'Initech'} — which Acme is suing.
      bridge path: Globex -> Initech


The conclusion ("Globex, owned by Initech, whom Acme is suing") is never stated in any one source
chunk — it *emerges* from traversing edges that came from three different documents. That edge-
following is exactly what GraphRAG's local search adds over similarity-only retrieval.

### 5.2 Global search: communities → summaries → a corpus-wide answer

Now a "sensemaking" question: *"What are the main themes in this corpus?"* We build a larger entity
graph, run **community detection** (greedy modularity — a CPU-friendly stand-in for the Leiden
algorithm Microsoft uses), and produce a one-line **summary per community**. Global search answers
by combining those summaries, never touching the raw text.

In [3]:
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

# A corpus's entities, grouped (loosely) into three latent themes.
edges = [
    # ML / training cluster
    ("transformer", "attention"), ("attention", "softmax"), ("transformer", "GPU"),
    ("GPU", "CUDA"), ("transformer", "pretraining"), ("pretraining", "dataset"),
    # Finance cluster
    ("Acme", "revenue"), ("revenue", "Q3"), ("Acme", "Globex"), ("Globex", "Initech"),
    ("Acme", "lawsuit"), ("lawsuit", "Initech"),
    # Climate cluster
    ("emissions", "CO2"), ("CO2", "warming"), ("warming", "policy"),
    ("policy", "carbon_tax"), ("emissions", "carbon_tax"),
]
KG = nx.Graph()
KG.add_edges_from(edges)

communities = sorted(greedy_modularity_communities(KG), key=len, reverse=True)
print(f"{KG.number_of_nodes()} entities -> {len(communities)} communities\n")

def summarize(community):
    # Stand-in for the LLM-written community report: rank members by degree.
    members = sorted(community, key=lambda n: KG.degree(n), reverse=True)
    hub = members[0]
    return hub, members

reports = []
for i, com in enumerate(communities):
    hub, members = summarize(com)
    report = f"Community {i} (theme hub: {hub!r}): " + ", ".join(members)
    reports.append((i, hub, report))
    print(report)

18 entities -> 3 communities

Community 0 (theme hub: 'transformer'): transformer, attention, pretraining, GPU, CUDA, softmax, dataset
Community 1 (theme hub: 'Acme'): Acme, Initech, lawsuit, revenue, Globex, Q3
Community 2 (theme hub: 'warming'): warming, carbon_tax, policy, emissions, CO2


In [4]:
# Global search = map-reduce over the community reports.
question = "What are the main themes in this corpus?"

# MAP: score each community report against the question. A real system asks the LLM
# "does this report help answer Q, and what's the partial answer?"; here we just keep
# all non-trivial communities and label them by their hub entity.
partials = [f"- theme around {hub!r} ({len(reports)} communities total)"
            for i, hub, _ in reports]

# REDUCE: combine partial answers into one. (LLM call in production; templated here.)
print(f"Q: {question}\n")
print("Global answer (reduced from community reports):")
print(f"The corpus covers {len(reports)} main themes, organised around the hub entities "
      + ", ".join(repr(h) for _, h, _ in reports) + ".")
print("\nMAP partials:")
print("\n".join(partials))

Q: What are the main themes in this corpus?

Global answer (reduced from community reports):
The corpus covers 3 main themes, organised around the hub entities 'transformer', 'Acme', 'warming'.

MAP partials:
- theme around 'transformer' (3 communities total)
- theme around 'Acme' (3 communities total)
- theme around 'warming' (3 communities total)


The global answer is a property of the *whole* graph — no chunk contains "the corpus has three
themes." Vanilla RAG, which can only retrieve chunks similar to the question, has nothing to
retrieve for "main themes" and would fail or hallucinate. The pre-computed community reports are
what make this cheap at query time: you read a handful of summaries instead of the entire corpus.

### 5.3 Where the triples come from: LLM entity extraction (gated)

§5.1's triples were hand-written. In a real pipeline an LLM reads each chunk and emits them. This
cell uses Claude when `ANTHROPIC_API_KEY` is set; otherwise it prints the call shape and uses a
trivial regex fallback so the notebook still runs.

In [5]:
import os, re, json

chunk = ("Globex is a supplier to Acme. Globex is owned by Initech. "
         "Acme is suing Initech over a patent dispute.")

EXTRACTION_PROMPT = (
    "Extract entities and relationships from the text as JSON: a list of "
    '{"source","relation","target"} triples. Text:\n\n' + chunk
)

if os.getenv("ANTHROPIC_API_KEY"):
    from anthropic import Anthropic  # pip install anthropic

    client = Anthropic()
    resp = client.messages.create(
        model="claude-haiku-4-5",   # cheap model is plenty for extraction
        max_tokens=512,
        messages=[{"role": "user", "content": EXTRACTION_PROMPT}],
    )
    print("LLM-extracted triples:\n", resp.content[0].text)
else:
    print("[no ANTHROPIC_API_KEY — showing call shape + regex fallback]\n")
    print("would POST to messages.create(model='claude-haiku-4-5', ...) with prompt:")
    print("  ", EXTRACTION_PROMPT.replace(chunk, "<chunk>"))
    # Naive fallback so the cell still produces triples deterministically.
    fallback = re.findall(r"(\w+) is (?:a )?(supplier to|owned by|suing) (\w+)", chunk)
    triples = [{"source": s, "relation": r.replace(" ", "_"), "target": t}
               for s, r, t in fallback]
    print("\nregex-extracted triples:")
    print(json.dumps(triples, indent=2))

[no ANTHROPIC_API_KEY — showing call shape + regex fallback]

would POST to messages.create(model='claude-haiku-4-5', ...) with prompt:
   Extract entities and relationships from the text as JSON: a list of {"source","relation","target"} triples. Text:

<chunk>

regex-extracted triples:
[
  {
    "source": "Globex",
    "relation": "supplier_to",
    "target": "Acme"
  },
  {
    "source": "Globex",
    "relation": "owned_by",
    "target": "Initech"
  },
  {
    "source": "Acme",
    "relation": "suing",
    "target": "Initech"
  }
]


## 6. Gotchas & Pitfalls

- **Indexing cost is the headline cost.** Extraction is at least one LLM call per chunk — for a
  large corpus that is thousands of calls *before you answer a single question*. Budget it, batch
  it, and use a cheap model for extraction. This is why GraphRAG is overkill for small or
  local-lookup workloads.
- **Garbage extraction → garbage graph.** The graph is only as good as the LLM's triples.
  Inconsistent entity names ("Acme", "Acme Corp", "ACME") fragment the graph; you need an **entity
  resolution / canonicalisation** step or your traversals dead-end. This is the #1 quality killer.
- **Re-indexing on updates is painful.** Add a document and you may need to re-extract, re-cluster,
  and re-summarise affected communities. Graphs are not as cheap to incrementally update as adding
  a vector to an index.
- **Global search burns tokens at query time too.** Map-reduce over many community summaries can
  itself be many LLM calls per question. Use the community hierarchy: answer from high-level
  summaries first, drill down only when needed.
- **It is not a vector-RAG replacement.** For local factoid lookups, GraphRAG is usually *worse*
  per dollar than a good embedding retriever. Most production systems run it **alongside** vector
  RAG, routing global/multi-hop questions to the graph.
- **Community detection is non-deterministic / parameterised.** Leiden has a resolution parameter;
  results shift with it. Don't treat community boundaries as ground truth.

## 7. When to Use vs Alternatives

| Approach | What it retrieves over | Strength | Cost / trade-off | Best when |
|---|---|---|---|---|
| **Vanilla vector RAG** | Flat text chunks by embedding similarity | Cheap, fast, accurate for local lookups | Misses multi-hop & global questions | Most QA over docs; the default |
| **GraphRAG (local search)** | Entity neighbourhood in a knowledge graph | Connects facts across documents (multi-hop) | LLM call per chunk to index | Relationships between entities matter |
| **GraphRAG (global search)** | LLM-written community summaries | Answers corpus-wide "sensemaking" questions | Index-time clustering + query-time map-reduce | "What are the themes / trends?" |
| **Hybrid (vector + reranker)** | Chunks, re-scored by a cross-encoder | Better precision than vector alone | Extra rerank latency | Local QA where top-k ordering matters |
| **Long-context stuffing** | The whole corpus in one prompt | No retrieval machinery at all | Quadratic cost; corpus must fit context | Small corpora that fit the window |

**Rule of thumb:** start with vanilla RAG. Add GraphRAG only when (a) users ask multi-hop or
corpus-wide questions, or (b) the *relationships* between entities are the product. When you do,
keep vanilla RAG for local queries and route by question type — see the
[`rag`](./rag.ipynb), [`rerankers`](./rerankers.ipynb), and
[`vector-db-comparison`](./vector-db-comparison.ipynb) notebooks for the local-retrieval half.

## 8. Resources

- **Edge et al. (2024), *From Local to Global: A Graph RAG Approach to Query-Focused
  Summarization*** — the Microsoft Research paper that defined the approach and the global
  map-reduce search: https://arxiv.org/abs/2404.16130
- **Microsoft GraphRAG — official docs & code** (the `graphrag` library, indexing/query CLI):
  https://microsoft.github.io/graphrag/ and https://github.com/microsoft/graphrag
- **Neo4j — *The GraphRAG Manifesto*** (graph-database vendor's practical take on knowledge-graph
  RAG and entity resolution): https://neo4j.com/blog/graphrag-manifesto/
- **NetworkX — community detection docs** (the algorithms used in §5.2):
  https://networkx.org/documentation/stable/reference/algorithms/community.html
- **Traag et al. (2019), *From Louvain to Leiden: guaranteeing well-connected communities*** — the
  clustering algorithm Microsoft GraphRAG uses: https://arxiv.org/abs/1810.08473